# 05 — Final Comparison & Submission
Rangkum semua model, pilih model terbaik, dan generate submission.

In [ ]:
import sys
sys.path.insert(0, '..')

import numpy as np
import json
import os

from src.utils import set_seed
from src.data import load_train, load_test
from src.cleaning import DataCleaner
from src.preprocessing import Preprocessor
from src.algorithms.decision_tree import DecisionTreeCART
from src.algorithms.logistic_regression import LogisticRegressionScratch
from src.algorithms.svm import LinearSVMScratch
from src.optimizers import Adam
from src.evaluation import cross_validate, macro_f1_score, find_best_threshold
from src.sklearn_baselines import get_sklearn_dtl, get_sklearn_lr, get_sklearn_svm
from src.visualization import plot_comparison_table
from src.predict import generate_submission
from src import config

set_seed(42)

In [ ]:
# Load saved CV results
results_path = os.path.join(config.ARTIFACTS_DIR, 'cv_results.json')
if os.path.exists(results_path):
    with open(results_path) as f:
        results = json.load(f)
    print('Loaded saved results:')
    for model in ['DTL', 'LR', 'SVM']:
        print(f'  {model}: scratch={results[model]["scratch"]:.4f}, sklearn={results[model]["sklearn"]:.4f}')
    print(f'\nBest model: {results["best_model"]}')
else:
    print('No saved results found. Run src/run_full_cv.py first.')

In [ ]:
# Comparison plot
comparison = {
    'Decision Tree (CART)': {'scratch': results['DTL']['scratch'], 'sklearn': results['DTL']['sklearn']},
    'Logistic Regression': {'scratch': results['LR']['scratch'], 'sklearn': results['LR']['sklearn']},
    'Linear SVM': {'scratch': results['SVM']['scratch'], 'sklearn': results['SVM']['sklearn']},
}
plot_comparison_table(
    comparison,
    title='Model Comparison: From-Scratch vs Sklearn (Macro F1, 5-Fold CV)',
    save_path=os.path.join(config.FIGURES_DIR, 'model_comparison.png')
)

In [ ]:
# Bar chart comparison
import matplotlib.pyplot as plt

models = ['CART', 'LR', 'SVM']
scratch_scores = [results['DTL']['scratch'], results['LR']['scratch'], results['SVM']['scratch']]
sklearn_scores = [results['DTL']['sklearn'], results['LR']['sklearn'], results['SVM']['sklearn']]

x = np.arange(len(models))
width = 0.35

fig, ax = plt.subplots(figsize=(10, 6))
bars1 = ax.bar(x - width/2, scratch_scores, width, label='From-Scratch', color='#3498db')
bars2 = ax.bar(x + width/2, sklearn_scores, width, label='Sklearn', color='#e74c3c')

ax.set_ylabel('Macro F1 Score')
ax.set_title('Model Comparison: From-Scratch vs Sklearn', fontweight='bold')
ax.set_xticks(x)
ax.set_xticklabels(models)
ax.legend()
ax.set_ylim(0.75, 0.90)

for bar in bars1 + bars2:
    height = bar.get_height()
    ax.annotate(f'{height:.4f}', xy=(bar.get_x() + bar.get_width()/2, height),
               xytext=(0, 3), textcoords='offset points', ha='center', va='bottom', fontsize=9)

plt.tight_layout()
plt.savefig(os.path.join(config.FIGURES_DIR, 'model_comparison_bar.png'), dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Generate final submission with best model (CART)
print('=== Generating Final Submission ===')
train_df = load_train()
test_df = load_test()

cleaner = DataCleaner()
train_clean = cleaner.fit_transform(train_df)
test_clean = cleaner.transform(test_df)

preprocessor = Preprocessor()
X_train, y_train = preprocessor.fit_transform(train_clean)
X_test = preprocessor.transform(test_clean)
test_ids = test_df[config.ID_COL].values

# Train CART on full training data
final_model = DecisionTreeCART(
    max_depth=10, min_samples_split=10, min_samples_leaf=5,
    class_weight='balanced'
)
final_model.fit(X_train, y_train)

# Generate submission
sub_path = generate_submission(final_model, X_test, test_ids, filename='submission.csv')
print(f'\nSubmission file: {sub_path}')

In [ ]:
# Verify submission format
import pandas as pd
sub = pd.read_csv(sub_path)
sample = pd.read_csv(config.SAMPLE_SUBMISSION_PATH)
print(f'Submission shape: {sub.shape} (expected {sample.shape})')
print(f'Columns match: {list(sub.columns) == list(sample.columns)}')
print(f'person_id match: {set(sub["person_id"]) == set(sample["person_id"])}')
print(f'\nPrediction distribution:')
print(sub['loan_status'].value_counts())